# 第 2 課：訓練一個「剪刀石頭布」分類器

第 1 課我們用 MNIST 走完一次流程，但 28x28 的手寫數字太小，
小到 NPU 只發揮了 **5.5%** 的實力。這一課換一個真的能用相機玩的題目。

**這一課結束時你會有：**

| 產出 | 用途 |
|---|---|
| `rps_cnn.h5` | 訓練好的模型，第 3 課拿去轉檔 |
| `rps_export.zip` | 量化校正用的 30 張圖 + `dataset.txt` + `channel_mean_value.txt` |
| 一組你自己量到的數字 | 準確率、參數量、MAC 數 |

**跟第 1 課不一樣的四件事**（這才是這一課真正在教的）：

1. 資料不是現成的 numpy，要自己下載、自己 resize
2. 這個資料集有個**很大的坑**，不先講你一定會踩
3. 要做 **data augmentation**，而且不能用一般教學那種寫法
4. 最後我們會**手算**這個模型要 NPU 出多少力，然後預測它跑多快

---
**執行方式**：從上往下，每一格按 `Shift + Enter`。不要跳格。

In [ ]:
# ---- 環境檢查 ----------------------------------------------------
import sys
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print("python       :", sys.version.split()[0])
print("tensorflow   :", tf.__version__)
print("keras        :", keras.__version__)
print("numpy        :", np.__version__)

# 固定亂數種子,讓你每次重跑拿到一樣的結果。
# 沒有這行,你改了一個參數之後分不出來是「改對了」還是「運氣好」。
SEED = 1234
tf.keras.utils.set_random_seed(SEED)

IMG = 96          # 模型輸入邊長,後面會解釋為什麼是 96
NUM_CLASSES = 3
CLASS_NAMES = ["rock", "paper", "scissors"]   # 石頭、布、剪刀(順序是資料集定的)

print()
print("輸入尺寸     :", (IMG, IMG, 3))
print("類別         :", CLASS_NAMES)

## 1. 先講這個資料集的坑

我們用 Laurence Moroney 做的 `rock_paper_scissors`，
**2520 張訓練 + 372 張測試**，300x300 PNG。

下一格會直接從 Google 的網址抓兩個 zip 下來解開，
**不走 `tensorflow_datasets`** —— 那個套件會拖進一串相依，
Colab 環境一更新就容易版本打架。直接讀資料夾比較穩，
而且那正是你之後拿自己拍的照片來訓練時要用的做法。

**坑在這裡：這些手不是拍出來的，是 3D 算圖算出來的。**

- 每張都是**純白背景**
- 光線永遠一致
- 手的材質是 CG 材質，不是真皮膚

所以你等一下會看到準確率衝得很漂亮，**那個數字會騙你**。
拿去對著真的相機，背景有雜物、光線不均、手的顏色不同 —— 準確率會掉。

**為什麼還是用它？**

1. 免費、乾淨、一行就下載得到，適合把整條流程先走通
2. 它的 test set 用的是**不同的手**，所以訓練/測試的落差是真的落差，看得出來
3. 等流程都通了，換成自己拍的資料只是換掉 `x_train` —— 這一課的程式碼完全不用改

**第 4 步的 augmentation 就是對抗這個坑的手段。**
它沒辦法完全補救（augmentation 生不出真背景），
但能讓模型少依賴「白底」這個作弊線索。

In [ ]:
# ---- 下載資料集(第一次跑 1~2 分鐘,之後有快取就很快)-----------------
# 刻意「不用」tensorflow_datasets。理由有兩個:
#   1. tfds 會拖進 tensorflow_metadata / protobuf 這串相依,
#      Colab 三不五時就會版本打架(protobuf runtime 比 gencode 舊 -> 直接 import 失敗)。
#   2. 你之後要用「自己拍的照片」訓練,那就是從資料夾讀圖 —— 就是下面這段。
#      現在學會了,換資料時只要改 ROOT 就好。
import os, zipfile, urllib.request
from PIL import Image

URLS = {
    "train": ("https://storage.googleapis.com/download.tensorflow.org/data/rps.zip",
              "rps"),
    "test":  ("https://storage.googleapis.com/download.tensorflow.org/data/rps-test-set.zip",
              "rps-test-set"),
}

def fetch(tag):
    url, root = URLS[tag]
    zf = tag + ".zip"
    if not os.path.exists(zf):
        print("下載 %s ..." % url)
        urllib.request.urlretrieve(url, zf)
        print("  %.0f MB" % (os.path.getsize(zf) / 1024.0 / 1024))
    if not os.path.isdir(root):
        with zipfile.ZipFile(zf) as z:
            z.extractall(".")
    return root

def load_folder(root):
    """從 root/<類別名>/*.png 讀圖。資料夾名稱 = 類別名稱,順序由 CLASS_NAMES 決定。"""
    paths, labels = [], []
    for ci, name in enumerate(CLASS_NAMES):
        d = os.path.join(root, name)
        assert os.path.isdir(d), "找不到資料夾 " + d
        for fn in sorted(os.listdir(d)):
            if fn.lower().endswith(".png"):
                paths.append(os.path.join(d, fn))
                labels.append(ci)

    # 先看第一張決定尺寸,再一次配好記憶體(比 append 到 list 再轉省一半 RAM)
    probe = Image.open(paths[0]).convert("RGB")
    H, W = probe.size[1], probe.size[0]
    x = np.empty((len(paths), H, W, 3), np.uint8)
    for i, p in enumerate(paths):
        # .convert("RGB") 很重要:這些 PNG 帶 alpha 通道(RGBA,4 通道),
        # 不轉的話 shape 會變成 (...,4),後面 assert 會當場爆給你看。
        x[i] = np.asarray(Image.open(p).convert("RGB"), np.uint8)
    return x, np.array(labels, np.int64)

x_train, y_train = load_folder(fetch("train"))
x_test,  y_test  = load_folder(fetch("test"))

print()
print("類別名稱    :", CLASS_NAMES, "(順序是我們自己定的,不是別人決定的)")
print("x_train     :", x_train.shape, x_train.dtype)
print("y_train     :", y_train.shape, y_train.dtype)
print("x_test      :", x_test.shape,  x_test.dtype)
print("佔記憶體    : %.0f MB" % (x_train.nbytes / 1024.0 / 1024))

In [ ]:
# ---- 先看資料,再訓練。這個順序不要顛倒 -----------------------------
print("像素範圍    :", x_train.min(), "~", x_train.max())
print("通道數      :", x_train.shape[-1], "(RGB)")
print()
print("每一類各有幾張:")
for i, name in enumerate(CLASS_NAMES):
    n_tr = int((y_train == i).sum())
    n_te = int((y_test  == i).sum())
    print("  %d %-9s train %4d   test %3d" % (i, name, n_tr, n_te))

# 類別平不平衡很重要:如果某一類特別多,模型只要一直猜那類就有不錯的準確率,
# 你會被準確率這個數字騙。這個資料集是刻意做成平均的。
cnt = np.bincount(y_train)
print()
print("最多 / 最少 = %.3f   (1.0 代表完全平均)" % (cnt.max() / cnt.min()))

In [ ]:
# ---- 每一類看 4 張 -------------------------------------------------
fig, axes = plt.subplots(3, 4, figsize=(10, 8))
for r, name in enumerate(CLASS_NAMES):
    idx = np.where(y_train == r)[0][:4]
    for c in range(4):
        ax = axes[r][c]
        ax.imshow(x_train[idx[c]])
        ax.axis("off")
        if c == 0:
            ax.set_title("%d  %s" % (r, name), loc="left", fontsize=12)
fig.suptitle("rock_paper_scissors  (raw 300x300 RGB)", fontsize=13)
plt.tight_layout(); plt.show()

# 注意看:背景全部是純白,手的角度變化也不大。這就是上面說的坑。

## 2. 為什麼縮到 96x96

原圖是 300x300x3。**不能直接拿去訓練**，三個理由，每個都跟板子有關：

**① 記憶體**
300x300x3 = 270,000 個值。第一層 conv 之後是 300x300x16 = 144 萬個值，
光一張 feature map 就吃掉 1.4 MB（int8）。AMB82 的 NPU 沒有那麼多工作記憶體。

**② 算力**
MAC 數跟邊長的平方成正比。300 換成 96，算力直接降到 **(96/300)² = 10.2%**。

**③ 相機本來就會幫你縮**
韌體裡的 `img_resize_planar()` 會把相機影像縮成模型要的尺寸。
所以你訓練時用多大，板子上就給你多大 —— 這件事是你決定的，不是相機決定的。

**那為什麼不是 64 或 128？**

| 邊長 | MAC（用我們的架構） | 手勢還看得清楚嗎 |
|---|---|---|
| 48 | 7.7 M | 剪刀和布開始分不出來 |
| **96** | **30.7 M** | **夠，這是我們的選擇** |
| 128 | 54.5 M | 更好，但算力多 78% |
| 224 | 167 M | MobileNetV2 用這個，對我們太重 |

96 還有一個好處：**96 → 48 → 24 → 12 → 6**，整除四次，
四層 pooling 下來不會出現小數，不用補 padding。這種數字選起來很省事。

In [ ]:
# ---- 300x300 縮到 96x96 -------------------------------------------
def resize_all(x):
    # tf.image.resize 回傳 float,轉回 uint8 省記憶體(augmentation 時再轉 float)
    out = tf.image.resize(x, (IMG, IMG), method="bilinear")
    return tf.cast(tf.round(out), tf.uint8).numpy()

mb_before = x_train.nbytes / 1024 / 1024
x_train = resize_all(x_train)
x_test  = resize_all(x_test)
mb_after = x_train.nbytes / 1024 / 1024

print("x_train     :", x_train.shape, x_train.dtype)
print("x_test      :", x_test.shape,  x_test.dtype)
print()
print("訓練集記憶體: %.1f MB  ->  %.1f MB   (省了 %.0f%%)"
      % (mb_before, mb_after, 100 * (1 - mb_after / mb_before)))

# 縮完再看一次,確認手勢還認得出來
fig, axes = plt.subplots(1, 6, figsize=(12, 2.4))
for i, ax in enumerate(axes):
    k = np.where(y_train == i % 3)[0][i // 3]
    ax.imshow(x_train[k]); ax.axis("off")
    ax.set_title(CLASS_NAMES[i % 3], fontsize=10)
fig.suptitle("after resize to %dx%d" % (IMG, IMG), fontsize=12)
plt.tight_layout(); plt.show()

## 3. 輸入契約（第 1 課講過，這裡確認一次）

板子那邊有兩個死規定，訓練時就要對齊，不然轉檔轉得出來、跑起來是垃圾：

**① 一定要 3 通道**

韌體的 `img_resize_planar()` 寫死輸出 `W * H * 3` bytes，
`img_t` 結構裡根本沒有通道數這個欄位。
如果你訓練一個 1 通道的模型，板子還是會塞 3 通道進去 ——
96x96x1 = 9,216 bytes 的緩衝區被寫進 27,648 bytes，**直接 overflow**。

這一課用 RGB，天生就是 3 通道，所以這關自動過。
（第 1 課的 MNIST 是灰階，我們得手動把它複製成 3 通道。）

**② `/255` 這件事，訓練時做、板子上不做**

我們在訓練時把像素從 0~255 變成 0~1。但板子餵進來的是原始的 0~255。
這個落差**不是靠程式碼補**，是靠轉檔時的 `channel_mean_value.txt`：

```
0 0 0 0.00392157
^ ^ ^ ^
| | | └── scale = 1/255
└─┴─┴──── mean（R G B 三個通道都減 0）
```

acuity 會把這個 scale 編進模型的第一層。所以**訓練用什麼正規化，那個檔就要寫什麼**。
寫錯了不會報錯，只會讓準確率莫名其妙變差 —— 這是最難抓的一種 bug。

In [ ]:
# ---- 把契約用 assert 釘住,違反就當場爆 -----------------------------
assert x_train.shape[1:] == (IMG, IMG, 3), "輸入必須是 %dx%dx3" % (IMG, IMG)
assert x_train.dtype == np.uint8,          "這裡要先保持 uint8"
assert x_train.min() >= 0 and x_train.max() <= 255

NORM_SCALE = 1.0 / 255.0
print("通道數      : %d  OK" % x_train.shape[-1])
print("正規化      : x / 255")
print("對應 channel_mean_value.txt -> 0 0 0 %.8f" % NORM_SCALE)

## 4. Data augmentation：這一課的重點

資料集只有 2520 張，而且全是白底。模型很容易學到**作弊的線索**，
例如「白色像素佔比」而不是「手指形狀」。Augmentation 就是在每個 epoch
把同一張圖用不同方式弄歪一點，逼模型去看真正重要的東西。

### 但是：不能用 `layers.RandomFlip` 那種寫法

Keras 有 `layers.RandomFlip` / `RandomRotation` 這類**前處理層**，
很多教學會叫你直接疊進模型裡。**對我們不行。**

| | 放在模型裡（`layers.RandomFlip`） | 放在資料管線裡（`tf.data.map`） |
|---|---|---|
| 訓練時 | 有作用 | 有作用 |
| 存成 .h5 | **層會被存進去** | 不會，模型很乾淨 |
| acuity 轉檔 | **看不懂這種層，轉檔失敗** | 完全沒問題 |

所以規矩是：**augmentation 只能存在於資料管線，不能存在於模型。**
模型裡只能有 conv / pool / dense / relu / softmax 這些 NPU 認得的東西。

（`Dropout` 和 `BatchNormalization` 是例外，可以放模型裡 ——
Dropout 推論時自動關掉，BatchNorm 會被 acuity 摺進前一層的權重。等一下會用到 Dropout。）

### 我們做哪些變化，為什麼

| 手法 | 參數 | 為什麼對這個題目有用 |
|---|---|---|
| 左右翻轉 | 50% | 資料集全是右手。翻轉之後左撇子也認得 |
| 亮度 | ±40 | 真實環境光線不會像 CG 一樣完美 |
| 對比 | 0.7 ~ 1.3 | 同上 |
| 隨機平移 | ±6 px | 手不會永遠正中央 |

**沒做旋轉**，因為 `tf.image` 沒有內建隨機旋轉（要裝 tensorflow-addons）。
這是你可以自己加的第一個實驗。

In [ ]:
# ---- augmentation 與資料管線 ---------------------------------------
AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 32
PAD = 6                     # 隨機平移的幅度(像素)

def augment(img, label):
    img = tf.cast(img, tf.float32)                       # 先轉 float,避免 uint8 溢位
    img = tf.image.random_flip_left_right(img)
    img = tf.image.random_brightness(img, max_delta=40.0)
    img = tf.image.random_contrast(img, lower=0.7, upper=1.3)
    # 先把圖補大一圈,再隨機裁回原尺寸 == 隨機平移
    img = tf.image.resize_with_crop_or_pad(img, IMG + 2 * PAD, IMG + 2 * PAD)
    img = tf.image.random_crop(img, [IMG, IMG, 3])
    img = tf.clip_by_value(img, 0.0, 255.0)              # 亮度/對比可能衝出範圍
    return img * NORM_SCALE, label                       # 最後才 /255

def plain(img, label):
    return tf.cast(img, tf.float32) * NORM_SCALE, label  # 驗證/測試不做 augmentation

def make_ds(x, y, training):
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    if training:
        ds = ds.shuffle(len(x), seed=SEED, reshuffle_each_iteration=True)
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    else:
        ds = ds.map(plain, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

# 從訓練集切 10% 出來當驗證集。
# 為什麼不用 test set 當驗證?因為那樣你等於在偷看考卷調參數,
# 最後 test 的分數就不再是誠實的分數了。
n_val = int(len(x_train) * 0.1)
rng = np.random.RandomState(SEED)
perm = rng.permutation(len(x_train))
val_idx, tr_idx = perm[:n_val], perm[n_val:]

ds_tr  = make_ds(x_train[tr_idx],  y_train[tr_idx],  training=True)
ds_val = make_ds(x_train[val_idx], y_train[val_idx], training=False)
ds_te  = make_ds(x_test,           y_test,           training=False)

print("訓練    : %4d 張  (有 augmentation)" % len(tr_idx))
print("驗證    : %4d 張  (沒有)" % len(val_idx))
print("測試    : %4d 張  (沒有,而且是不同的手)" % len(x_test))
print("batch   : %d" % BATCH_SIZE)

In [ ]:
# ---- 親眼看 augmentation 做了什麼:同一張圖跑 7 次 --------------------
one = x_train[np.where(y_train == 2)[0][0]]        # 挑一張剪刀

fig, axes = plt.subplots(1, 8, figsize=(15, 2.3))
axes[0].imshow(one); axes[0].axis("off"); axes[0].set_title("original", fontsize=10)
for i in range(1, 8):
    aug, _ = augment(one, 0)
    axes[i].imshow(np.clip(aug.numpy(), 0, 1))
    axes[i].axis("off"); axes[i].set_title("aug %d" % i, fontsize=10)
fig.suptitle("same image, 7 random augmentations "
             "(model never sees it twice the same way)", fontsize=12)
plt.tight_layout(); plt.show()

## 5. 模型架構

| 層 | 輸出尺寸 | 在做什麼 |
|---|---|---|
| Input | 96 x 96 x 3 | |
| Conv2D 16, 3x3 + ReLU | 96 x 96 x 16 | 找邊緣、顏色變化 |
| MaxPool 2x2 | 48 x 48 x 16 | 尺寸砍半，留最強的訊號 |
| Conv2D 32, 3x3 + ReLU | 48 x 48 x 32 | 把邊緣組成角、弧 |
| MaxPool 2x2 | 24 x 24 x 32 | |
| Conv2D 64, 3x3 + ReLU | 24 x 24 x 64 | 組成「指尖」「手掌輪廓」這種東西 |
| MaxPool 2x2 | 12 x 12 x 64 | |
| Conv2D 64, 3x3 + ReLU | 12 x 12 x 64 | 組成整個手勢 |
| MaxPool 2x2 | 6 x 6 x 64 | |
| **Reshape (2304,)** | 2304 | 攤平 |
| Dropout 0.3 | 2304 | 訓練時隨機關掉 30%，防止死背 |
| Dense 64 + ReLU | 64 | |
| Dense 3 + Softmax | 3 | 三個機率，加起來 = 1 |

### 三個為什麼

**為什麼 `Reshape` 不是 `Flatten`？**
`Flatten` 產生的是**動態**形狀（「不管前面多大我都攤平」）。
NPU 在編譯時就要把每一塊記憶體的位置釘死，遇到動態形狀它算不出來。
`Reshape((2304,))` 把數字寫死，NPU 才知道要配多少記憶體。
這是第 1 課踩過的坑，這裡直接用對的寫法。

**為什麼四層 conv，不是兩層？**
每經過一次 3x3 conv + pool，一個神經元「看得到」的原圖範圍（感受野）就放大一倍。
四層下來，最後一層每個點大約看得到原圖 **60x60** 的範圍 —— 差不多是整隻手。
兩層的話只看得到 ~14x14，只夠看到一根手指，分不出剪刀和布。

**為什麼通道數是 16 → 32 → 64 → 64？**
慣例是「尺寸砍半時通道加倍」，讓每層的計算量大致持平。
最後一層沒有再加倍到 128，是因為 Dense 層的參數量會爆掉
（12x12x64 攤平是 9216，乘上 Dense 64 就是 59 萬個參數）。
我們靠多一層 pooling 把它壓到 2304 才攤平。

In [ ]:
# ---- 建模型 --------------------------------------------------------
def build_model():
    inp = keras.Input(shape=(IMG, IMG, 3), name="input")
    x = layers.Conv2D(16, 3, padding="same", activation="relu", name="conv1")(inp)
    x = layers.MaxPooling2D(2, name="pool1")(x)
    x = layers.Conv2D(32, 3, padding="same", activation="relu", name="conv2")(x)
    x = layers.MaxPooling2D(2, name="pool2")(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu", name="conv3")(x)
    x = layers.MaxPooling2D(2, name="pool3")(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu", name="conv4")(x)
    x = layers.MaxPooling2D(2, name="pool4")(x)
    x = layers.Reshape((6 * 6 * 64,), name="reshape")(x)     # 寫死,不用 Flatten
    x = layers.Dropout(0.3, name="dropout")(x)               # 推論時自動失效,可以安心用
    x = layers.Dense(64, activation="relu", name="dense1")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax", name="output")(x)
    return keras.Model(inp, out, name="rps_cnn")

model = build_model()
model.compile(optimizer=keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

In [ ]:
# ---- 自己算一次參數量,跟 summary 對答案 ----------------------------
# 算得出來,才代表你真的知道每一層在存什麼東西。
def conv_params(kh, kw, cin, cout):  return kh * kw * cin * cout + cout
def dense_params(nin, nout):         return nin * nout + nout

rows = [
    ("conv1",  conv_params(3, 3,   3, 16), "3x3x3   x16  + 16 bias"),
    ("conv2",  conv_params(3, 3,  16, 32), "3x3x16  x32  + 32 bias"),
    ("conv3",  conv_params(3, 3,  32, 64), "3x3x32  x64  + 64 bias"),
    ("conv4",  conv_params(3, 3,  64, 64), "3x3x64  x64  + 64 bias"),
    ("dense1", dense_params(2304, 64),     "2304    x64  + 64 bias"),
    ("output", dense_params(64, 3),        "64      x3   + 3  bias"),
]
total = 0
print("%-8s %10s   %s" % ("層", "參數量", "怎麼來的"))
print("-" * 52)
for name, n, how in rows:
    print("%-8s %10d   %s" % (name, n, how))
    total += n
print("-" * 52)
print("%-8s %10d" % ("合計", total))
print()
print("Keras 說 :", model.count_params())
assert total == model.count_params(), "算錯了"
print("對上了")
print()
print("Dropout / MaxPool / Reshape 都是 0 個參數 —— 它們只搬動資料,不記東西。")
print("注意 dense1 一層就佔了 %.0f%% 的參數。" % (100.0 * 147520 / total))
print("這就是為什麼要多一層 pooling 把 12x12 壓成 6x6 才攤平。")

## 6. 先看起跑線

**訓練之前先評估一次。** 這一步很多教學會跳過，但它是你唯一的對照組 ——
沒有它，你不知道模型到底學到東西了，還是只是在猜。

三個類別的隨機猜測應該是：

- accuracy ≈ **1/3 = 0.333**
- loss ≈ **ln(3) = 1.0986**

如果訓練前就不是這兩個數字，代表初始化或資料有問題，先停下來查。

In [ ]:
# ---- 訓練前的成績 --------------------------------------------------
loss0, acc0 = model.evaluate(ds_te, verbose=0)
print("訓練前 test loss : %.4f   (理論值 ln(3) = %.4f)" % (loss0, np.log(3)))
print("訓練前 test acc  : %.4f   (理論值 1/3  = %.4f)" % (acc0, 1.0 / 3))

# 把還沒訓練的 conv1 權重存起來,最後跟訓練後的比
W0_conv1 = model.get_layer("conv1").get_weights()[0].copy()
print()
print("conv1 初始權重 : shape %s   範圍 %.3f ~ %.3f"
      % (W0_conv1.shape, W0_conv1.min(), W0_conv1.max()))
print("(現在它是一堆亂數,等一下你會看到它變成真的有意義的圖案)")

## 7. 開始訓練

**epoch** = 把全部訓練資料完整看過一遍。

我們設 **20 個 epoch**，比第 1 課的 6 個多。原因有兩個：
資料量少（2268 張 vs MNIST 的 54000 張），而且有 augmentation ——
每個 epoch 看到的圖都不一樣，所以模型需要更多輪才學得完。

**訓練時盯著看的是 `val_accuracy`，不是 `accuracy`。**
`accuracy` 是模型對它看過的題目的表現，會一路漂亮下去；
`val_accuracy` 才是對沒看過的題目的表現。
兩者開始分岔的那一刻，就是開始死背的那一刻。

跑起來大概 2~4 分鐘（Colab CPU）。想快一點就切
**執行階段 → 變更執行階段類型 → T4 GPU**。

In [ ]:
# ---- 訓練 ----------------------------------------------------------
EPOCHS = 20

hist = model.fit(
    ds_tr,
    validation_data=ds_val,
    epochs=EPOCHS,
    verbose=2,
)
print()
print("訓練完成")

In [ ]:
# ---- 學習曲線 ------------------------------------------------------
h = hist.history
ep = np.arange(1, len(h["loss"]) + 1)

fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))

ax[0].plot(ep, h["loss"],     "o-", label="train loss")
ax[0].plot(ep, h["val_loss"], "s-", label="val loss")
ax[0].axhline(np.log(3), ls="--", c="gray", lw=1)
ax[0].text(1, np.log(3) + .02, "random guess ln(3)", fontsize=8, color="gray")
ax[0].set_xlabel("epoch"); ax[0].set_ylabel("loss")
ax[0].set_title("loss  (lower is better)"); ax[0].legend(); ax[0].grid(alpha=.3)

ax[1].plot(ep, h["accuracy"],     "o-", label="train acc")
ax[1].plot(ep, h["val_accuracy"], "s-", label="val acc")
ax[1].axhline(1.0 / 3, ls="--", c="gray", lw=1)
ax[1].text(1, 1.0 / 3 + .01, "random guess 1/3", fontsize=8, color="gray")
ax[1].set_ylim(0, 1.02)
ax[1].set_xlabel("epoch"); ax[1].set_ylabel("accuracy")
ax[1].set_title("accuracy  (higher is better)"); ax[1].legend(); ax[1].grid(alpha=.3)

plt.tight_layout(); plt.show()

In [ ]:
# ---- 同樣的資料,用表格看一次(圖看趨勢,表看數字)-------------------
print("%8s %11s %10s %10s %9s %9s" %
      ("epoch", "train loss", "val loss", "train acc", "val acc", "val 進步"))
print("-" * 64)
print("%8s %11.4f %10s %10.4f %9s %9s" % ("(訓練前)", loss0, "-", acc0, "-", "-"))
prev = None
for i in range(len(ep)):
    va = h["val_accuracy"][i]
    delta = "-" if prev is None else "%+.4f" % (va - prev)
    print("%8d %11.4f %10.4f %10.4f %9.4f %9s" %
          (ep[i], h["loss"][i], h["val_loss"][i], h["accuracy"][i], va, delta))
    prev = va

best = int(np.argmax(h["val_accuracy"]))
print("-" * 64)
print("最好的一輪  : epoch %d,  val acc %.4f" % (ep[best], h["val_accuracy"][best]))
gap = h["accuracy"][-1] - h["val_accuracy"][-1]
print("最後一輪落差: train %.4f - val %.4f = %+.4f" %
      (h["accuracy"][-1], h["val_accuracy"][-1], gap))
print()
if gap > 0.10:
    print("落差 > 0.10:開始死背了。加強 augmentation 或提高 Dropout。")
elif h["val_accuracy"][-1] < 0.80:
    print("val acc < 0.80:還學不夠。加 epoch 或把模型加大。")
else:
    print("落差和準確率都在合理範圍。")

## 8. 打開模型看它學到什麼

`conv1` 有 16 個 3x3x3 的濾鏡。訓練前是亂數，
訓練後應該變成**看得出規律**的圖案 —— 邊緣偵測器、顏色對比偵測器之類的。

如果訓練完還是一片雜訊，代表這一層根本沒學到東西（通常是學習率設太大）。

In [ ]:
# ---- conv1 的 16 個濾鏡:訓練前 vs 訓練後 ---------------------------
W1_conv1 = model.get_layer("conv1").get_weights()[0]

def draw_filters(W, ax_row):
    for i in range(16):
        f = W[:, :, :, i]
        f = (f - f.min()) / (f.max() - f.min() + 1e-9)   # 拉到 0~1 才看得到
        ax_row[i].imshow(f); ax_row[i].axis("off")

fig, axes = plt.subplots(2, 16, figsize=(16, 2.8))
draw_filters(W0_conv1, axes[0])
draw_filters(W1_conv1, axes[1])
axes[0][0].set_title("before training (random)", loc="left", fontsize=11)
axes[1][0].set_title("after training", loc="left", fontsize=11)
plt.tight_layout(); plt.show()

print("權重分布:")
print("  訓練前  std = %.4f   範圍 %.3f ~ %.3f"
      % (W0_conv1.std(), W0_conv1.min(), W0_conv1.max()))
print("  訓練後  std = %.4f   範圍 %.3f ~ %.3f"
      % (W1_conv1.std(), W1_conv1.min(), W1_conv1.max()))
spread = W1_conv1.std() / W0_conv1.std()
print("  std 變成原來的 %.2f 倍" % spread)
print()
if spread > 1.05:
    print("std 變大 = 模型把有用的特徵放大、沒用的壓小,這是有在學的跡象。")
elif spread < 0.95:
    print("std 變小 = 權重被壓縮。不一定是壞事,但配合上面的曲線一起看。")
else:
    print("std 幾乎沒變。如果準確率也沒起來,通常是 epoch 太少或學習率太小。")
print()
print("不管變大變小,這個範圍在量化時很關鍵 —— 它直接決定 int8 的 scale:")
print("  scale = max|W| / 127 = %.6f" % (np.abs(W1_conv1).max() / 127))
print("範圍越大,每一格 int8 代表的數值就越粗,量化誤差就越大。")

In [ ]:
# ---- 看一張圖經過 conv1 之後變成什麼 -------------------------------
probe = keras.Model(model.input, model.get_layer("conv1").output)
sample = x_test[np.where(y_test == 2)[0][0]]          # 一張剪刀
fmap = probe.predict(sample[None].astype("float32") * NORM_SCALE, verbose=0)[0]

fig = plt.figure(figsize=(16, 2.6))
ax = fig.add_subplot(1, 9, 1); ax.imshow(sample); ax.axis("off")
ax.set_title("input (scissors)", fontsize=10)
for i in range(8):
    ax = fig.add_subplot(1, 9, i + 2)
    ax.imshow(fmap[:, :, i], cmap="viridis"); ax.axis("off")
    ax.set_title("filter %d" % i, fontsize=9)
plt.tight_layout(); plt.show()

print("亮的地方 = 這個濾鏡在那裡找到了它要找的東西。")
print("不同濾鏡亮的位置不一樣,代表它們各自負責不同的特徵。")

## 9. 考卷檢討：它到底錯在哪

準確率是一個數字，但它不告訴你**錯在哪裡**。
三類問題只有 6 種可能的錯法，全部列出來看。

注意：下面用的是 **test set**，不是 val set。
test set 的手跟訓練用的手不一樣，所以這才是誠實的分數。

In [ ]:
# ---- 混淆矩陣 ------------------------------------------------------
pred = model.predict(ds_te, verbose=0)
y_pred = pred.argmax(axis=1)
conf_all = pred.max(axis=1)

test_loss, test_acc = model.evaluate(ds_te, verbose=0)
print("test accuracy : %.4f   (%d / %d 答對)"
      % (test_acc, int((y_pred == y_test).sum()), len(y_test)))
print("val  accuracy : %.4f   (最後一輪)" % h["val_accuracy"][-1])
print()

cm = np.zeros((NUM_CLASSES, NUM_CLASSES), dtype=int)
for t, p in zip(y_test, y_pred):
    cm[t][p] += 1

print("            預測")
print("         " + "".join("%10s" % n for n in CLASS_NAMES) + "%8s" % "recall")
for i, name in enumerate(CLASS_NAMES):
    r = cm[i][i] / float(cm[i].sum())
    print("%-9s" % name + "".join("%10d" % v for v in cm[i]) + "%8.3f" % r)
print()
print("對角線 = 答對。其他格子 = 把左邊那類看成上面那類。")

In [ ]:
# ---- 混淆矩陣畫成圖 + 每類 recall -----------------------------------
fig, ax = plt.subplots(1, 2, figsize=(12, 4.4))

im = ax[0].imshow(cm, cmap="Blues")
ax[0].set_xticks(range(3)); ax[0].set_xticklabels(CLASS_NAMES)
ax[0].set_yticks(range(3)); ax[0].set_yticklabels(CLASS_NAMES)
ax[0].set_xlabel("predicted"); ax[0].set_ylabel("expected")
ax[0].set_title("confusion matrix")
for i in range(3):
    for j in range(3):
        ax[0].text(j, i, cm[i][j], ha="center", va="center",
                   color="white" if cm[i][j] > cm.max() * .5 else "black", fontsize=13)
fig.colorbar(im, ax=ax[0], fraction=.046)

rec = np.array([cm[i][i] / float(cm[i].sum()) for i in range(3)])
bars = ax[1].bar(CLASS_NAMES, rec, color=["#4c72b0", "#dd8452", "#55a868"])
ax[1].axhline(test_acc, ls="--", c="k", lw=1)
ax[1].text(2.45, test_acc + .012, "overall", fontsize=9, ha="right")
ax[1].set_ylim(0, 1.05); ax[1].set_ylabel("recall")
ax[1].set_title("per-class recall  (of N real X, how many found)")
for b, v in zip(bars, rec):
    ax[1].text(b.get_x() + b.get_width() / 2, v + .015, "%.3f" % v,
               ha="center", fontsize=10)
ax[1].grid(axis="y", alpha=.3)

plt.tight_layout(); plt.show()

print("六種錯法,由多到少:")
errs = [(cm[i][j], CLASS_NAMES[i], CLASS_NAMES[j])
        for i in range(3) for j in range(3) if i != j]
any_err = False
for n, a, b in sorted(errs, reverse=True):
    if n:
        print("  %3d 次  把 %-9s 看成 %s" % (n, a, b))
        any_err = True
if not any_err:
    print("  (沒有錯)")

In [ ]:
# ---- 預期 vs 實際:挑 24 張來看(綠色=對,紅色=錯)-------------------
rng2 = np.random.RandomState(SEED)
pick = rng2.choice(len(x_test), 24, replace=False)

fig, axes = plt.subplots(4, 6, figsize=(14, 10))
for ax_, k in zip(axes.ravel(), pick):
    ax_.imshow(x_test[k]); ax_.axis("off")
    ok = y_pred[k] == y_test[k]
    ax_.set_title("%s -> %s\n%.0f%%" %
                  (CLASS_NAMES[y_test[k]], CLASS_NAMES[y_pred[k]], conf_all[k] * 100),
                  fontsize=9, color=("green" if ok else "red"))
fig.suptitle("expected -> predicted   (green = correct, red = wrong)", fontsize=13)
plt.tight_layout(); plt.show()

In [ ]:
# ---- 最值得看的:它「很有信心」但答錯的那些 -------------------------
wrong = np.where(y_pred != y_test)[0]
print("總共答錯 %d 張" % len(wrong))

if len(wrong):
    order = wrong[np.argsort(-conf_all[wrong])]
    n = min(6, len(order))
    fig, axes = plt.subplots(1, n, figsize=(2.3 * n, 3))
    if n == 1:
        axes = [axes]
    for ax_, k in zip(axes, order[:n]):
        ax_.imshow(x_test[k]); ax_.axis("off")
        ax_.set_title("%s -> %s\n%.1f%% sure" %
                      (CLASS_NAMES[y_test[k]], CLASS_NAMES[y_pred[k]],
                       conf_all[k] * 100), fontsize=9, color="red")
    fig.suptitle("most confident mistakes", fontsize=12)
    plt.tight_layout(); plt.show()
    print()
    print("這些是最該看的樣本 —— 模型不只答錯,還很篤定。")
    print("通常代表訓練資料裡缺了這一種樣子的手。")
else:
    print("全對。這在這個 CG 資料集上是有可能的 —— 但別高興太早,")
    print("真相機的表現會差很多,因為那裡沒有純白背景。")

## 10. 這個模型要 NPU 出多少力

這一段是這一課跟第 1 課最大的不同：**在燒進板子之前，先算出它該跑多快。**

有了預測值，等你真的量到數字時才知道結果是好是壞。
沒有預測值，量到任何數字你都只能說「喔」。

**MAC** = 一次乘法加一次加法，是 NPU 的基本工作單位。

- Conv2D 的 MAC = `輸出高 x 輸出寬 x 輸出通道 x (kernel高 x kernel寬 x 輸入通道)`
- Dense 的 MAC = `輸入數 x 輸出數`

AMB82 的 NPU 標稱 0.4 TOPS @ 500 MHz，換算下來是 **400 MAC/cycle** 的理論上限。

In [ ]:
# ---- 手算 MAC ------------------------------------------------------
def conv_mac(oh, ow, cout, kh, kw, cin):  return oh * ow * cout * kh * kw * cin
def dense_mac(nin, nout):                 return nin * nout

layers_mac = [
    ("conv1",  conv_mac(96, 96, 16, 3, 3,  3), "96x96x16 x (3x3x3)"),
    ("conv2",  conv_mac(48, 48, 32, 3, 3, 16), "48x48x32 x (3x3x16)"),
    ("conv3",  conv_mac(24, 24, 64, 3, 3, 32), "24x24x64 x (3x3x32)"),
    ("conv4",  conv_mac(12, 12, 64, 3, 3, 64), "12x12x64 x (3x3x64)"),
    ("dense1", dense_mac(2304, 64),            "2304 x 64"),
    ("output", dense_mac(64, 3),               "64 x 3"),
]
total_mac = sum(m for _, m, _ in layers_mac)

print("%-8s %14s %7s   %s" % ("層", "MAC", "佔比", "怎麼來的"))
print("-" * 58)
for name, m, how in layers_mac:
    print("%-8s %14s %6.1f%%   %s"
          % (name, "{:,}".format(m), 100.0 * m / total_mac, how))
print("-" * 58)
print("%-8s %14s" % ("合計", "{:,}".format(total_mac)))
print()

MNIST_MAC  = 1342848          # 第 1 課那個模型
YOLOV7_MAC = 2.79e9           # yolov7-tiny @416,我們在板子上量過的

print("跟已知的模型比:")
print("  第 1 課 MNIST   %14s   (我們這個是它的 %.1f 倍)"
      % ("{:,}".format(MNIST_MAC), float(total_mac) / MNIST_MAC))
print("  本課 RPS        %14s" % "{:,}".format(total_mac))
print("  YOLOv7-tiny     %14s   (是我們這個的 %.0f 倍)"
      % ("{:,}".format(int(YOLOV7_MAC)), YOLOV7_MAC / total_mac))

In [ ]:
# ---- 從已知的量測值推算它會跑多快 ----------------------------------
# 這兩個數字是我們在 AMB82 上實際量到的(NbBenchSD,用 NPU 自己的 cycle counter)
MNIST_CYCLES  = 60900          # 1,342,848 MAC
YOLOV7_CYCLES = 31910000       # 約 2.79 G MAC
CLK_HZ        = 500e6
PEAK_MAC_PER_CYCLE = 400       # 0.4 TOPS @ 500 MHz

mnist_eff = MNIST_MAC  / float(MNIST_CYCLES)
yolo_eff  = YOLOV7_MAC / float(YOLOV7_CYCLES)

print("已知的 NPU 效率(同一顆晶片、同一個時脈、同一種量法):")
print("  MNIST        %6.1f MAC/cycle   利用率 %4.1f%%"
      % (mnist_eff, 100 * mnist_eff / PEAK_MAC_PER_CYCLE))
print("  YOLOv7-tiny  %6.1f MAC/cycle   利用率 %4.1f%%"
      % (yolo_eff, 100 * yolo_eff / PEAK_MAC_PER_CYCLE))
print()
print("模型越大利用率越高 —— 小模型餵不飽 NPU,光搬資料的開銷就吃掉大半。")
print()
print("我們這個模型夾在中間,所以預測值也夾在中間:")
print()
print("%-16s %12s %11s %9s" % ("假設效率", "cycles", "時間", "FPS"))
print("-" * 53)
for tag, eff in [("像 MNIST 一樣", mnist_eff),
                 ("兩者中間",      (mnist_eff + yolo_eff) / 2),
                 ("像 YOLOv7 一樣", yolo_eff)]:
    cyc = total_mac / eff
    us  = cyc / CLK_HZ * 1e6
    print("%-16s %12s %8.0f us %9.0f" % (tag, "{:,.0f}".format(cyc), us, 1e6 / us))
print("-" * 53)
print()
print("→ 預測:單次推論落在 0.7 ~ 3 ms 之間。")
print("  等你燒進板子用 NbBenchSD 量到真實 cycle 數,回來對這張表。")
print("  落在區間內 = 我們對這顆 NPU 的模型是對的。")
print("  落在區間外 = 有我們還沒搞懂的東西,那反而更值得追。")

## 11. 存檔，準備轉檔

轉檔（第 3 課）需要三樣東西，這一格全部產生好：

| 檔案 | 是什麼 | 為什麼需要 |
|---|---|---|
| `rps_cnn.h5` | 模型本體 | acuity 要讀它的結構和權重 |
| `calib/*.png` + `dataset.txt` | 30 張校正圖 | 量化時要知道每一層的數值範圍長什麼樣 |
| `channel_mean_value.txt` | `0 0 0 0.00392157` | 把訓練時的 `/255` 編進模型第一層 |

**校正圖為什麼要 30 張、為什麼從 test set 挑？**

量化就是把 float32 的權重和中間結果壓成 int8。要壓得準，acuity 得先知道
「實際跑的時候，每一層的數值大概在什麼範圍」。它靠的就是拿這幾張圖跑一遍去統計。

- 挑太少（1~2 張）→ 統計不準，量化誤差大
- 挑太多 → 轉檔變慢，沒有額外好處
- **從 test set 挑**，是因為要代表「模型上線後會看到的東西」，不是它背過的東西
- 三類**各挑 10 張**，不能全挑同一類

存檔時 `include_optimizer=False`：optimizer 的狀態（Adam 的動量之類）
只有繼續訓練才用得到，acuity 不需要，留著只會讓檔案變大、還可能讓它讀不懂。

In [ ]:
# ---- 產生轉檔需要的全部檔案 ----------------------------------------
import os, shutil
from PIL import Image

OUT = "rps_export"
shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(os.path.join(OUT, "calib"))

# 1) 模型
h5_path = os.path.join(OUT, "rps_cnn.h5")
model.save(h5_path, include_optimizer=False)
print("模型      : %s   (%.1f KB)" % (h5_path, os.path.getsize(h5_path) / 1024.0))

# 1b) tflite —— 第 3 課離線轉檔真正吃的是這個,不是 .h5
#     這裡刻意「不」量化,維持 float32。量化要留給 acuity 做,
#     因為只有它知道 NPU 要的量化格式長什麼樣。
tfl_path = os.path.join(OUT, "rps_cnn.tflite")
tfl = tf.lite.TFLiteConverter.from_keras_model(model).convert()
with open(tfl_path, "wb") as f:
    f.write(tfl)
print("tflite    : %s   (%.1f KB)" % (tfl_path, os.path.getsize(tfl_path) / 1024.0))

# 2) 校正圖:每類 10 張,從 test set 挑
PER_CLASS = 10
lines = []
for c in range(NUM_CLASSES):
    idx = np.where(y_test == c)[0][:PER_CLASS]
    for j, k in enumerate(idx):
        fn = "calib/%s_%02d.png" % (CLASS_NAMES[c], j)
        Image.fromarray(x_test[k]).save(os.path.join(OUT, fn))   # 已經是 RGB 3 通道
        lines.append("./" + fn)
print("校正圖    : %d 張  (%d 類 x %d 張)  %dx%dx3"
      % (len(lines), NUM_CLASSES, PER_CLASS, IMG, IMG))

# 3) dataset.txt:acuity 靠這個檔知道要拿哪些圖去統計
with open(os.path.join(OUT, "dataset.txt"), "w") as f:
    f.write("\n".join(lines) + "\n")

# 4) channel_mean_value.txt:R_mean G_mean B_mean scale
with open(os.path.join(OUT, "channel_mean_value.txt"), "w") as f:
    f.write("0 0 0 %.8f\n" % NORM_SCALE)
print("正規化    : 0 0 0 %.8f" % NORM_SCALE)

# 5) 一併存下這次訓練的成績,之後對照用
with open(os.path.join(OUT, "train_report.txt"), "w") as f:
    f.write("input        : %dx%dx3\n" % (IMG, IMG))
    f.write("classes      : %s\n" % ", ".join(CLASS_NAMES))
    f.write("params       : %d\n" % model.count_params())
    f.write("MAC          : %d\n" % total_mac)
    f.write("epochs       : %d\n" % EPOCHS)
    f.write("seed         : %d\n" % SEED)
    f.write("val accuracy : %.4f\n" % h["val_accuracy"][-1])
    f.write("test accuracy: %.4f\n" % test_acc)
print("成績單    : train_report.txt")

shutil.make_archive("rps_export", "zip", OUT)
print()
print("打包完成  : rps_export.zip  (%.1f KB)"
      % (os.path.getsize("rps_export.zip") / 1024.0))

In [ ]:
# ---- 下載到你的電腦(只有 Colab 能跑)------------------------------
from google.colab import files
files.download("rps_export.zip")

### 為什麼要存兩種格式

Colab 現在預設是 **TensorFlow 2.19 / Keras 3**，Realtek 的 acuity 工具鏈停在
**TensorFlow 2.14.1** 那個年代。Keras 3 寫出來的 `.h5` 裡面是新的 layer config，
acuity 讀不懂 —— 而且它**不會在 Colab 這裡報錯**，是等你第 3 課轉檔時才炸。

我們已經踩過這個坑了。出路不是把 Colab 降版（會連鎖壞掉別的東西），
而是**改用 `.tflite`**：

| | `.h5` | `.tflite` |
|---|---|---|
| 格式穩不穩 | 跟著 Keras 版本一直變 | FlatBuffer，多年沒變過 |
| acuity 讀得懂嗎 | 看版本，常常不行 | 可以 |
| 裡面有什麼 | 結構 + 權重 + 一堆 Keras 自己的東西 | 只剩算圖和權重 |

所以上面那格**兩個都存**：`.tflite` 是第 3 課要用的，`.h5` 留著當備份
（想回頭改架構、接著訓練時還是得靠它）。

下面這格是**萬一**你真的需要退回 Keras 2 時的逃生門。
正常流程用不到，先別跑。

In [ ]:
# ---- 逃生門:正常流程不要跑這格 --------------------------------------
# 只有當你確定要用 .h5 餵 acuity、而且它抱怨 layer config 時才跑。
# 跑完 -> 選單「執行階段 -> 重新啟動工作階段」-> 回到最上面重跑
%pip install -q tf-keras
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"     # 一定要在 import tensorflow 之前
print("設定好了。現在去重新啟動執行階段,然後從第 1 格重跑。")

## 12. 這一課你做了什麼

1. 下載真實資料集，而且**先認清它的限制**（CG 白底）再開始訓練
2. 把 300x300 縮成 96x96，理由是記憶體、算力、以及「反正相機會幫你縮」
3. 學會 **augmentation 要放在資料管線、不能放在模型** —— 這是能不能轉檔成功的分水嶺
4. 建了一個四層 conv 的模型，並且**手算參數量對過答案**
5. 先量起跑線（1/3、ln3），才知道學到的東西是真的
6. 用 test set（不同的手）誠實評分，不是用調過參的 val set
7. **手算 MAC 並預測 NPU 速度** —— 現在你有一個可以被推翻的預測

## 下一步

拿到 `rps_export.zip` 之後有兩條路：

| | 線上轉檔 | 離線 acuity |
|---|---|---|
| 怎麼做 | 傳到 amebaiot.com，等 email | 自己跑 7 步驟 |
| 快不快 | 等回信 | 幾分鐘 |
| 看得到細節嗎 | 看不到 | 每一步都看得到 |
| 會不會踩 TF 版本坑 | 會（上限 2.14.1） | 一樣會 |

**第 3 課會走離線那條**，因為只有那條你才看得到量化到底對準確率做了什麼。

## 自己動手（改一個變數，重跑，記錄結果）

做這四個實驗，你對訓練的直覺會比看十篇文章有用：

1. **關掉 augmentation** —— 把 `ds_tr` 那行的 `training=True` 改成 `False`。
   train acc 會更漂亮，test acc 會更差。親眼看到「死背」長什麼樣。
2. **`IMG = 48`** —— 算力剩 1/4，準確率掉多少？這是你的性能/精度取捨表第一列。
3. **拿掉 Dropout** —— 看 train 和 val 曲線的分岔提早多少。
4. **`EPOCHS = 60`** —— 找出 val accuracy 從哪一輪開始不再進步。
   那一輪之後的訓練全是浪費，而且會讓模型更會死背。

做完把數字記下來。**那張表比這份 notebook 更有價值**，因為它是你自己量的。

---

## 13. 後記：這個模型燒進板子之後發生了什麼

**這一節是第 2 課最重要的一節。** 上面那些你照著跑都會成功，
下面這些是我們真的把 .nb 燒進 AMB82、開了鏡頭之後，被現實打臉學到的。

### 13.1 test accuracy 1.0000 不是成績，是紅燈

我們這次訓練的成績單長這樣：

```
val accuracy : 1.0000
test accuracy: 1.0000
```

第一次看到會很爽。但你應該要不安，理由很簡單：

> **真實世界的分類問題不會 100%。** 一個模型在測試集上滿分，
> 通常不是它很強，是**測試集太簡單**，或是測試集跟訓練集太像。

這個資料集是 CG 算圖的手、**白底、光線一致、手的角度只有幾種**。
train / test 切開了沒錯，但兩邊都來自同一台算圖機。
模型只要學會「白底上那團膚色的輪廓」就能滿分 —— 它根本不需要真的理解手勢。

**怎麼判斷自己是不是中了這一招**：看 test accuracy 跟**實際部署**的差距。
我們的差距是 100% → 幾乎全錯。

### 13.2 Domain gap：訓練資料跟現場不是同一個世界

| | 訓練資料 | 我辦公室的 AMB82 |
|---|---|---|
| 背景 | 純白 | 桌面、螢幕、我的臉、牆 |
| 光線 | 均勻打光 | 頂燈 + 窗戶逆光 |
| 影像 | CG 算圖，邊緣乾淨 | 相機，有雜訊、有壓縮 |
| 手 | 算圖的標準手 | 我的手，距離忽遠忽近 |
| 顏色 | 標準 | 白平衡偏黃 |

這個落差有個名字叫 **domain gap**。它不是 bug，不是訓練沒跑夠，
也不是模型太小 —— **再訓練 1000 個 epoch 都不會變好**，
因為模型從頭到尾就沒看過現場長什麼樣。

**唯一的解法是讓訓練資料靠近現場。** 最有效的做法不是加 augmentation，
是**直接拿要部署的那台相機去拍資料**。這就是下一步要做的事。

### 13.3 softmax 沒有「我不知道」這個選項

這是我們花最久才想通的一件事。現象是：**鏡頭前面什麼都沒有，
板子還是很有自信地說 rock 99%。**

原因不在模型，在數學。模型最後一層是 softmax：

```
logits = [z_rock, z_paper, z_scissors]          # 三個實數
p_i    = exp(z_i) / ( exp(z_rock) + exp(z_paper) + exp(z_scissors) )
```

分母是三項的和，所以 **p_rock + p_paper + p_scissors 恆等於 1**。
不管你餵它什麼 —— 天花板、你的臉、一片黑 —— 這三個數字加起來永遠是 1。
**這個架構在結構上就沒有能力輸出「都不是」。** 它只能回答
「如果一定要三選一，最像哪一個」。

順帶一提，這也是為什麼**光加信心門檻不夠**：
神經網路對沒見過的東西（out-of-distribution）常常**錯得很有自信**，
softmax 值照樣飆到 0.99。門檻擋得掉一部分，擋不掉全部。

**兩個解法要一起用：**

1. **加第四類 none** —— 讓「沒有手」變成一個模型真的學過的答案。
   這是治本的，因為它把 OOD 問題變回一個普通的分類問題。
2. **加信心門檻** —— 最高分低於門檻就回報 unknown，不要判勝負。
   治標，但便宜，而且擋得掉模糊的中間狀態（手比到一半）。

### 13.4 對照組：分類、偵測、關鍵點

順便把三種做法的差別講清楚，你以後選型會用到：

| | 影像分類（我們現在） | 物件偵測（YOLO） | 手部關鍵點（MediaPipe） |
|---|---|---|---|
| 輸出 | 整張圖一個標籤 | 0~N 個框 + 信心 | 21 個座標，或 0 個 |
| 能說「沒有」嗎 | **不能**（見 13.3） | 能（回 0 個框） | 能（回 0 隻手） |
| 手在畫面哪裡 | 不知道 | 知道 | 知道 |
| 抗背景干擾 | 弱 | 強 | 強 |
| 標註成本 | 低（分資料夾就好） | **高**（每張畫框） | 零（不用訓練） |
| 模型大小 | 222 KB | yolov4-tiny 約 4.1 MB | 內建 |

我們留在分類，是因為**標註成本低、而且你要學的是「怎麼訓練」**。
分類的弱點（13.3）可以用加一類 + 門檻補掉，代價很小。

### 13.5 所以下一步

1. 用**這台 AMB82 的鏡頭**收真實照片（背景、光線、你的手，全都對了）
2. 加第四類 **none** —— 鏡頭前沒有手
3. 重新訓練，然後**期待 accuracy 掉下來** —— 掉下來才是真的
4. 板子端加**信心門檻**，低於門檻就回報 unknown

第 2.5 課（025_retrain_rps.ipynb）就是做這件事。
這次你會自己收資料 —— **收資料的品質決定模型的上限，比調參數重要十倍。**